In [10]:
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

In [11]:
# 1) Load data
df = pd.read_csv("dataset_v2_full_features.csv")

# 2) Cari kolom target secara case-insensitive
target_candidates = [c for c in df.columns if c.lower() == "amr_status" or c.lower() == "amr_status".lower()]
if not target_candidates:
    raise ValueError("Kolom target AMR_Status / AMR_status tidak ditemukan di dataset.")

target_col = target_candidates[0]
X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

# 3) Siapkan tipe fitur
categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

# 4) Encode label target
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 5) Split train-test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
 )

# 6) Preprocessing + model boosting yang lebih ringan
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
 )

boost_model = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_depth=8,
    max_iter=200,
    min_samples_leaf=20,
    l2_regularization=0.1,
    random_state=42
 )

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", boost_model),
])

# 7) Train
pipeline.fit(X_train, y_train)

# 8) Evaluasi
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Target column: {target_col}")
print(f"Accuracy: {acc:.4f}\n")

print("Classification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Target column: AMR_Status
Accuracy: 0.9991

Classification Report:
              precision    recall  f1-score   support

Intermediate       1.00      0.99      0.99       153
   Resistant       1.00      1.00      1.00       534
 Susceptible       1.00      1.00      1.00      1653

    accuracy                           1.00      2340
   macro avg       1.00      1.00      1.00      2340
weighted avg       1.00      1.00      1.00      2340

Confusion Matrix:
[[ 151    2    0]
 [   0  534    0]
 [   0    0 1653]]


In [12]:
# Tambahkan setelah pipeline.fit(...)
import joblib
import numpy as np

# 1) Simpan pipeline lengkap (recommended)
joblib.dump(pipeline, "amr_hgb_pipeline.joblib")

# 2) Simpan label target (biar hasil prediksi bisa dibalik ke nama kelas)
np.save("amr_label_classes.npy", label_encoder.classes_)

print("Model tersimpan.")

Model tersimpan.
